# M6 — Hyperparameter Tuning, Evaluasi Test Set, Analisis Kesalahan & Pemilihan Model Final
**Proyek:** Klasifikasi Penyakit Daun Padi (*Rice Leaf Disease Classification*)

Notebook ini menampilkan hasil evaluasi akhir ketiga algoritma yang telah dituning pada **Hold-Out Test Set (962 citra unik dari 172 spesimen daun fisik yang belum pernah dilihat saat training)**, lengkap dengan *Confusion Matrix*, analisis *Overfitting/Underfitting*, dan *Feature Importance*.

In [1]:
import json
import sys
from pathlib import Path
import pandas as pd
from IPython.display import Image as IPImage, display

sys.path.append(str(Path.cwd().parent))
from src import REPORTS_DIR, FIGURES_DIR

df_comp = pd.read_csv(REPORTS_DIR / "model_comparison.csv")
print("=== Tabel Perbandingan Akhir pada Hold-Out Test Set (962 Citra) ===")
cols = ["model", "cv_f1_macro", "accuracy", "precision_macro", "recall_macro", "f1_macro", "roc_auc_ovr", "train_f1_macro", "model_size_mb"]
print(df_comp[cols].to_string(index=False))

=== Tabel Perbandingan Akhir pada Hold-Out Test Set (962 Citra) ===
        model  cv_f1_macro  accuracy  precision_macro  recall_macro  f1_macro  roc_auc_ovr  train_f1_macro  model_size_mb
    SVM (RBF)       0.9761    0.9875           0.9853        0.9889    0.9867       1.0000             1.0           0.28
Random Forest       0.9540    0.9751           0.9733        0.9733    0.9733       0.9996             1.0           1.43
      XGBoost       0.9716    0.9875           0.9894        0.9844    0.9865       0.9996             1.0           0.38


In [2]:
# Menampilkan Grafik Perbandingan Model, Confusion Matrix, dan Feature Importance
for fig_name in ["05_model_comparison.png", "06_confusion_matrices.png", "07_feature_importance.png"]:
    print("Visualisasi:", fig_name)
    display(IPImage(filename=str(FIGURES_DIR / fig_name)))

Visualisasi: 05_model_comparison.png
Visualisasi: 06_confusion_matrices.png
Visualisasi: 07_feature_importance.png


### Kesimpulan Evaluasi & Alasan Pemilihan Model Final (M6)
1. **Model Terbaik (Model Final): `SVM (RBF)`** dipilih karena memiliki skor **5-Fold Group-CV F1-Macro tertinggi pada data latih (`0,9761`)**, dan saat diuji satu kali pada *hold-out test set* mencapai **Accuracy `98,75%`**, **F1-Macro `0,9867`**, **Recall Macro `0,9889`**, serta **ROC-AUC `1,0000`** dengan ukuran file paling ringan (**`0,28 MB`**).
2. **Perbandingan dengan XGBoost & Random Forest:**
   - **XGBoost** memberikan akurasi uji yang setara dengan SVM (`98,75%`, F1-Macro `0,9865`, ukuran `0,38 MB`).
   - **Random Forest** berada sedikit di bawah keduanya (`97,51%`, F1-Macro `0,9733`, ukuran `1,43 MB`).
3. **Analisis Overfitting:** Selisih (*gap*) antara skor Train (`1,0000`) dan Test (`0,9867` pada SVM dan `0,9865` pada XGBoost) sangat kecil (`~0,013`), membuktikan bahwa model mampu melakukan generalisasi dengan sangat baik terhadap spesimen daun baru yang belum pernah dilihat.
4. **Analisis Kesalahan (*Error Analysis*):** Kelas `Brownspot` (240/240) dan `Tungro` (260/260) berhasil diprediksi **100% sempurna** oleh ketiga model. Kesalahan minor (12 citra dari 962 citra uji, atau 1,25%) hanya terjadi antara `Bacterialblight` dan `Blast` karena kedua penyakit tersebut pada fase lanjut sama-sama memiliki bercak nekrotik pucat memanjang.